# 固定案例：仿射价格数量与真实份额/现金的会计对接
3个按事件类型/日期选择的100股单位案例；非零B仅符号公式（可用分钟域内无该事件）。0策略回放/拟合/账户/抓取/订单，先审计划再比较数值。

In [1]:
from pathlib import Path
import os,json,qlib
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
card=root/'research/affine_raw_accounting_bridge_20261015';plan=json.loads((card/'plan.json').read_text())
qlib.init(provider_uri=str(root/plan['provider_uri']),region='us',kernels=1)
from src.workflow.affine_raw_accounting_bridge import run
summary=run(root=root,card=card,plan=plan)
print(json.dumps(summary,ensure_ascii=False,indent=2))

[82154:MainThread](2026-10-15 20:05:27,923) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


2026/10/15 20:05:28 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[82154:MainThread](2026-10-15 20:05:28,382) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[82154:MainThread](2026-10-15 20:05:28,382) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/full_path_tiered_allocation_20261013')}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{
  "experiment": "affine_raw_accounting_bridge_20261015",
  "plan_sha256": "1e03bf5f8b8f813a46e2cb26909c8d2d1cec4759693bcb2e7d160db935bf75c6",
  "decision": "finite_unit_bridge_only_dividend_cash_and_PIT_need_data",
  "objective_achieved": false,
  "numeric_unit_cases": 3,
  "Qlib_price_anchor_rows": 8,
  "Qlib_features_calls": 8,
  "Qlib_calendar_calls": 1,
  "symbolic_nonzero_B_identity": "ForconstantA,B and formulaP_adj=A*P_raw+B: q_adj*P_adj_t -n*P_raw_t = n*B*(P_raw0-P_raw_t)/(A*P_raw0+B). SymbolicnonzeroB implication only; no empiricalnonzeroBprices availableinbounds.",
  "nonzero_B_scope": {
    "SOXL": [
      "2010-09-22"
    ],
    "SOXX": []
  },
  "nonzero_B_empirical_case_available": false,
  "case_gates": [
    {
      "case": "stable_operator_control",
      "source_identity": "passed_pinned_source_and_anchors",
      "unit_identity": "passed_algebra_and_provider_roundoff",
      "event_accounting": "not_applicable_no_event",
      "PIT_timing": "not_qualified_historica

In [2]:
# Reporting-only correction approved after original execution; no new market reads.
import hashlib,pandas as pd
from src.core.recorder_sources import load_verified_recorder_objects
objects,audit=load_verified_recorder_objects(card,expected_recorder_id='b6cf662c0feb4d46badbdad4ccf18346',expected_manifest_sha256='b7b4ef23e5a6aa55d391d177dfbb4e37b5e9cae4ce6653fdd7c143650eda8d1d',artifacts=['analysis/cases','analysis/marks','analysis/cash_fields'])
original=objects['analysis/cases'];corrected=original.copy();mask=corrected['case'].eq('split_declared_no_cash')
assert mask.sum()==1 and not bool(corrected.loc[mask,'implied_raw_share_identity_within_tolerance'].iloc[0])
before=corrected.loc[mask,'event_accounting'].iloc[0]
corrected.loc[mask,'event_accounting']='split_quantity_rounding_exceeds_frozen_tolerance'
pd.testing.assert_frame_equal(original.drop(columns='event_accounting'),corrected.drop(columns='event_accounting'))
corrected.to_csv(card/'case_gates_corrected.csv',index=False)
manifest=json.loads((card/'verification_manifest.json').read_text())
source_hashes={e['path'].split('/artifacts/')[1]:e['sha256'] for e in manifest['entries'] if e['path'].split('/artifacts/')[-1] in ['analysis/cases','analysis/marks','analysis/cash_fields','analysis/summary','sources/execution_workflow']}
cell=next(c for c in json.loads((card/(card.name+'.ipynb')).read_text())['cells'] if c['id']=='corrected_reporting')
receipt={'role':'post_execution_reporting_only','original_recorder_id':'b6cf662c0feb4d46badbdad4ccf18346','original_archive_sha256':'1c6646b9b6cbb091029e691afc75c3ab224b6cc34669b8d6f08d492632e29c7a','source_artifact_hashes':source_hashes,'cell_source_sha256':hashlib.sha256(''.join(cell['source']).encode()).hexdigest(),'case':'split_declared_no_cash','field':'event_accounting','before':before,'after':'split_quantity_rounding_exceeds_frozen_tolerance','reason':'Published eventA0.33333 versus exact split1/3 causes fixedlot implied300.00300003 versus physical300; this exceeds frozen quantity tolerance. Not an unknown cash-ledger cause. Qlib float32 residual remains separately reported.','gate_booleans_changed':False,'tolerances_changed':False,'new_Qlib_price_reads':0,'new_unit_cases':0,'new_replays':0,'original_source_summary_objects_unchanged':True}
(card/'reporting_correction.json').write_text(json.dumps(receipt,indent=2)+'\n')
print(corrected[['case','event_accounting','implied_raw_share_identity_within_tolerance']].to_string(index=False))


                        case                                    event_accounting  implied_raw_share_identity_within_tolerance
     stable_operator_control                             not_applicable_no_event                                         True
      split_declared_no_cash    split_quantity_rounding_exceeds_frozen_tolerance                                        False
cash_dividend_multiplicative needs_data_currency_entitlement_paydate_cash_ledger                                        False


# 有限单位会计对接：数量、现金与时钟必须分开

执行固定3个100股单位案例，读取8个既有Qlib 09:47锚点和一次原生日历。案例按事件类型/日期选择，不按收益选取；0模型训练、政策回放、新账户、行情/公司行动抓取、订单或策略净值/风险重算。

|案例|单位数量结果|资格解释|
|---|---|---|
|SOXL 2023-01-04→01-05 稳定因子|隐含真实份额100，与固定100股一致|无公司行动时单位映射一致，不产生前向时钟资格|
|SOXX 2024-03-07 拆股|公布A=0.33333；确切份额比1→3；映射得300.00300003股而非300股|超过预冻结数量容差，差异来自公布拆股因子精度；不扩容差补救|
|SOXL 2023-03-21 分红|A=0.99849、B=0；事件后隐含100.15122835股，固定真实份额仍100股|价格代理隐含风险资产敞口与不再投资的真实份额不同；现金/应收款资格仍不足|

Qlib float32读回误差另行报告，没有与公布因子舍入或事件敞口差异混为一谈。上述数字是100股单位案例，不是策略损益偏差、全历史失效证据或真实券商账本认证。

SOXL语料唯一非零B事件为2010-09-22，早于2018-10-12起的可用分钟域；SOXX无非零B事件。本次2023+案例的B均为0，因此非零B只保留符号恒等式，没有捏造经验案例。A不等于1不能作为拆股分类器：现金分红也可能通过A调整。

原始字段per_cash_div=0.02501明确单独保留，100股条件金额为2.501个来源现金单位。未验证币种、权益认定、公告/支付时点及实际收款账本，故没有把该数值加入美元净值、赋予可交易现金或假定自动再投资，也没有从B反推现金分红。现金到账与应收款不是同一状态。

所有行情/事件接收证据都是2026-10-12历史检索，不能证明案例日决策前已收到输入；右边界分钟条及可用时间假设也不是实际接收时钟。来源身份和代数校验通过，不代表事件现金会计或PIT时序通过。

**报告修正：**首次执行的通用分类分支误把拆股数量容差失败标成现金账本缺口。原Recorder、执行源码、summary、manifest和NAS原包保持不变；独立审查批准仅在已保存对象上更正报告标签。有效报告见`case_gates_corrected.csv`，修正来源哈希与代码cell哈希见`reporting_correction.json`。Notebook的事后代码cell仅读取原始cases/marks/cash_fields；未新增价格查询、案例、回放、容差或通过项。拆股仍为失败，理由改为split_quantity_rounding_exceeds_frozen_tolerance。

决定finite_unit_bridge_only_dividend_cash_and_PIT_need_data。下一步最小依赖是对固定SOXL 2023-03-21分配记录做有限的既有发行人语料核对：金额币种、权益/记录日、支付日和原始披露/接收身份；不能用A/B代替这些字段。任何真实现金/拆股处理合同或价格基准修复都需新身份与独立计划，不修改封存27baa，也不自动启动观察器。历史rolling/CI失败不变。

Notebook完成原始8锚点检查及一次仅报告重导出。没有额外软件测试或策略验证。独立评估已确认数值并批准报告修正；NAS冷恢复逐一核验19个对象哈希一致；独立最终评估接受有效修订报告。
